# 9 · Forced alignment — phone boundaries in time

Stage 6 tells you **what** was said. This stage tells you **when** each phone
starts and ends. The output is a Praat **TextGrid** per utterance, with a word
tier and a phone tier — the format Praat, ELAN and every TTS front-end expect.

The aligner is the Kölsch model from stage 5 plus
`torchaudio.functional.forced_align`. It needs no pronunciation dictionary
beyond `kolsch_g2p.py`, which is the point: MFA and MAUS both need a German
lexicon, and **94.1 % of Kölsch word tokens are out of vocabulary** against the
152,766-word `german_mfa` dictionary.

---

## Start here: `absorb="vc"`

CTC is **peaky**. The model emits one confident frame per phone and blanks in
between, so on the reference material the labelled frames cover only **14–21 %**
of the timeline — the demo below measures its own figure and prints it, and on the shipped one-speaker example it comes out nearer 31 %. Roughly **four fifths of every phone duration you see in a TextGrid is
not measured — it is a rule deciding who gets the blank frames.**

Which rule you pick therefore matters more than the model does, and the obvious
rules are all wrong in the same way. They assume each CTC spike sits in the
middle of its phone. It does not:

| | spike sits … through its segment |
|---|---|
| vowels | **71 %** (late) |
| consonants | **10 %** (early) |

*(measured against MFA over one recording and its two halves, n = 27 vowels /
35 consonants; unstable in detail — the consonant median is 29 % on the full
file and 6 % on the chunks — but the direction is stable)*

So the blank run between a consonant spike and the following vowel spike starts
at the **beginning** of the consonant and ends **two thirds into** the vowel.
Split it down the middle and the consonant eats half the vowel. That is exactly
what you see: `/h/` in *høːt* came out **307 ms** where MFA said 10 ms, and
`/b/` in *bɛsɐ* took **197 ms** while its own vowel kept 42.

`absorb="vc"` gives each word-internal blank run to the **vowel**. Word onsets
are untouched, so cross-system comparisons stay valid.

| gap rule | three-way spread (median) | all three within 50 ms |
|---|---|---|
| `hybrid` | 72.9 ms | 38.2 % |
| **`vc`** | **67.4 ms** | **41.4 %** |

Over 84 field recordings and 3,742 phones: **68 improve, 15 get worse**, median
change −3.0 ms. Read the per-class split before believing the headline — every
vocalic class improves, every consonantal class is flat or very slightly worse.
The two nearly cancel, and "vc is better" is a **net** claim, not a uniform one.


In [ ]:
!pip -q install torch torchaudio transformers librosa soundfile pandas matplotlib
import torch, numpy as np, pandas as pd, librosa
from pathlib import Path
print("torch", torch.__version__, "·",
      "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)


## The model

Weights are **not** in this repository — the fine-tuned checkpoint is ~1.2 GB,
well past what git should carry. Point `MODEL_DIR` at either

* the folder stage 5 wrote into `models/`, or
* a Hugging Face repo id, once you have pushed the checkpoint there.

`PROCESSOR_DIR` may be the same folder; it is separate here only because the
reference checkpoint keeps the tokenizer beside the weights.


In [ ]:
from kolsch_align import (Aligner, MODES, write_textgrid, VOCALIC,
                          SAMPLE_RATE)

# Either a local directory (stage 5 output) or a Hugging Face repo id.
MODEL_DIR     = os.environ.get("KOLSCH_MODEL",     os.path.join(MODELS, "kolsch_wav2vec2_model_all"))
PROCESSOR_DIR = os.environ.get("KOLSCH_PROCESSOR", MODEL_DIR)

al = Aligner(MODEL_DIR, PROCESSOR_DIR, lexicon=LEXICON)
print(f"{len(al.vocab)} symbols · blank id {al.blank_id} · "
      f"{sum(p.numel() for p in al.model.parameters())/1e6:.0f}M params · {al.device}")
print("modes:", ", ".join(MODES))


## Phone classes

`vc` needs to know which symbols are vowels. The table lives in `kolsch_align.py`
so notebooks 9 and 9b cannot disagree about it. **A symbol missing from it is
treated as a consonant**, which silently disables the rule for that phone — so
the `Aligner` refuses to build if your vocabulary contains one it cannot class.


In [ ]:
from kolsch_align import (VOWEL_SHORT, VOWEL_LONG, DIPHTHONG, PLOSIVE,
                          AFFRICATE, FRICATIVE, NASAL, APPROXIMANT, KNOWN)

for name, s in (("vowel_short", VOWEL_SHORT), ("vowel_long", VOWEL_LONG),
                ("diphthong", DIPHTHONG), ("plosive", PLOSIVE),
                ("affricate", AFFRICATE), ("fricative", FRICATIVE),
                ("nasal", NASAL), ("approximant", APPROXIMANT)):
    print(f"  {name:12} {' '.join(sorted(s))}")
print("\nunclassified in this model's vocabulary:",
      {s for s in al.syms if s not in KNOWN} or "none")


## Text → phone chain

Dictionary first (`data/lexicon.csv`, human-checkable), the rule-based converter
for anything not in it — the same policy as stage 4. The IPA string is then cut
into the model's symbols by longest-match, so `t͡s`, `aɪ` and `øː` stay whole
instead of decomposing into their parts.


In [ ]:
# Dictionary first (data/lexicon.csv, human-checkable), the rule-based converter
# for anything not in it -- the same policy as stage 4. The IPA string is then cut
# into the model's symbols by longest match, so t͡s, aɪ and øː stay whole.
print(f"{len(al.lex)} lexicon entries\n")
for w in ("Kind", "d'r", "jewonnt"):
    print(f"  {w:10} -> {' '.join(al.ipa_to_phones(al.lex.get(al._clean(w)) or ''))
                          or ' '.join(al.text_to_chain(w)[0][0])}")
chain, words = al.text_to_chain("Als Kind han ich")
print("\nchain:", chain, "\nwords:", words)


## The gap rules

Four modes, all operating on the same CTC output. Only the treatment of the
blank runs differs.

| mode | what happens to a blank run |
|---|---|
| `none` | nothing — phones keep only their labelled frames, and the TextGrid has holes |
| `even` | split down the middle |
| `hybrid` | cut at the **spectral-change peak** inside the word; posterior-weighted across word edges |
| **`vc`** | word-internally, C→V and V→C runs go to the **vowel**; everything else as `hybrid` |

**The one exception in `vc`,** and it is fitted rather than derived: an
*intervocalic* fricative or affricate keeps the flux peak. Its left boundary is
already its own spike (from the V→C rule), so giving the right-hand run away too
would leave it a single 20 ms frame — the `/s/` of *bɛsɐ* collapsed to 20 ms
against MFA's 210 and MAUS's 190. A fricative's spike marks the *onset* of
frication and the noise continues; a stop's spike sits at the burst. Phonetics
supports the distinction, **this data does not prove it** — `/s/` in *bɛsɐ* is
the only instance in the material. Revisit if it ever misfires.


In [ ]:
# The rules live in kolsch_align.absorb_gaps() so this notebook and 9b cannot
# drift apart. Read it here rather than trusting the table above.
import inspect
from kolsch_align import absorb_gaps, pause_span_in_gap, spectral_flux, energy_envelope

print(inspect.getsource(absorb_gaps))


## Aligning one utterance

`torchaudio.functional.forced_align` is a Viterbi pass over the CTC lattice: it
returns the single most likely frame-to-token path **given the phone chain you
supply**. It cannot skip, insert or reorder a phone. If the chain is wrong the
alignment is confidently wrong, which is why stage 4 exists and why the words
come from a transcript rather than from the model's own decode.


In [ ]:
# al.align() is kolsch_align.Aligner.align -- forced_align plus the chosen rule.
print(inspect.getsource(al.align.__func__))


## TextGrid output\n\nPraat short-text format, two interval tiers. Opens in Praat and ELAN unchanged.

In [ ]:
# write_textgrid is imported from kolsch_align; holes become empty intervals,
# which is what Praat requires and what MFA and MAUS emit for silence.
print(inspect.getsource(write_textgrid))


## Demo — what `vc` actually changes

One utterance, aligned four ways. The table below is the same phone chain every
time; only the gap rule differs, so any duration that moves is the rule moving
it, not the model.


In [ ]:
MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), "run notebook 3 first — it writes data/segments/manifest.csv"
man = pd.read_csv(MANIFEST)
print(len(man), "segments")

row = man.iloc[0]
print("text:", row["text"])

runs = {m: al.align(row["audio_path"], row["text"], mode=m)[0]
        for m in ("none", "even", "hybrid", "vc", "vc-sil")}

cmp = pd.DataFrame({
    "phone": [p["label"] for p in runs["vc"]],
    **{f"{m} (ms)": [round((p["end"] - p["start"]) * 1000) for p in runs[m]]
       for m in ("none", "even", "hybrid", "vc", "vc-sil")},
})
cmp["class"] = ["vowel" if p in VOCALIC else "cons." for p in cmp["phone"]]
cmp["vc − hybrid"] = cmp["vc (ms)"] - cmp["hybrid (ms)"]
display(cmp)

lab = sum(p["end"] - p["start"] for p in runs["none"])
tot = runs["vc"][-1]["end"] - runs["vc"][0]["start"]
print(f"\nCTC labelled {lab:.2f}s of a {tot:.2f}s span — {100*lab/tot:.0f}%. "
      f"The other {100-100*lab/tot:.0f}% is the gap rule.")
for c in ("vowel", "cons."):
    d = cmp.loc[cmp["class"] == c, "vc − hybrid"]
    print(f"  {c:6s} n={len(d):3d}  mean change {d.mean():+6.1f} ms")
print("\nNotebook 9b measures this across every segment, plus the pause case.")


Vowels should gain and consonants should lose. If that is not what the two lines above say, the phone-class table is wrong for your inventory.

In [ ]:
import matplotlib.pyplot as plt

def draw(ax, phones, y, colour, h=0.42):
    for p in phones:
        ax.add_patch(plt.Rectangle((p["start"], y), p["end"] - p["start"], h,
                                   facecolor=colour, alpha=0.22, edgecolor=colour, lw=0.9))
        ax.annotate(p["label"], ((p["start"] + p["end"]) / 2, y + h / 2),
                    ha="center", va="center", fontsize=8)

wav, _ = librosa.load(row["audio_path"], sr=SAMPLE_RATE)
dur = len(wav) / SAMPLE_RATE
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(np.arange(len(wav)) / SAMPLE_RATE, wav / (np.abs(wav).max() or 1) * 0.4 + 0.55,
        lw=0.4, color="#c8c7c2")
for phones, name, col, y in ((runs["hybrid"], "hybrid", "#9dc2ea", -0.50),
                             (runs["vc"], "vc", "#2a78d6", -1.05)):
    draw(ax, phones, y, col)
    ax.annotate(name, (-0.008, y + 0.21), xycoords=("axes fraction", "data"),
                ha="right", va="center", fontsize=9, fontweight="bold", color=col)
for a, b in zip(runs["hybrid"][1:], runs["vc"][1:]):        # every boundary that moved
    if abs(a["start"] - b["start"]) > 0.005:
        ax.annotate("", xy=(b["start"], -1.05 + 0.42), xytext=(a["start"], -0.50),
                    arrowprops=dict(arrowstyle="->", color="#990011", lw=1.0, alpha=0.75))
ax.set_xlim(0, dur); ax.set_ylim(-1.25, 1.05); ax.set_yticks([])
ax.set_xlabel("time (s)"); ax.set_title(row["text"], loc="left", fontsize=10)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()


## Batch — a TextGrid for every segment

In [ ]:
TG_DIR = os.path.join(DATA, "textgrids"); os.makedirs(TG_DIR, exist_ok=True)
ABSORB = "vc"          # "vc-sil" if you want silence emitted -- see notebook 9b

ok = fail = 0
for _, r in man.iterrows():
    stem = Path(r["audio_path"]).stem
    try:
        phones, words, dur = al.align(r["audio_path"], r["text"], mode=ABSORB)
        write_textgrid(os.path.join(TG_DIR, f"{stem}.TextGrid"), dur,
                       [("words", words), ("phones", phones)])
        ok += 1
    except Exception as e:
        fail += 1
        print(f"  {stem}: {type(e).__name__}: {e}")
print(f"{ok} TextGrids ({ABSORB}) -> {TG_DIR}" + (f"   ({fail} failed)" if fail else ""))


---

## Comparing against MFA and MAUS

Neither runs here, and both are **optional**. They are documented because the
`vc` numbers quoted at the top are agreement figures against these two, and an
agreement figure you cannot reproduce is not a result.

Give all three systems the **same phone chain** — otherwise you are measuring
three different pronunciation dictionaries, not three aligners.

### MFA (local, offline)

```bash
conda install -c conda-forge montreal-forced-aligner   # 2.2.17 here
mfa model download acoustic german_mfa
mfa model download dictionary german_mfa
mfa align data/segments/ german_mfa german_mfa out/mfa/
```

Runs entirely on your machine. Expect a large OOV rate on Kölsch — 94.1 % of
tokens against the pretrained dictionary — so supply your own lexicon built from
`kolsch_g2p.py` if you want the comparison to be fair.

### MAUS (BAS webservice)

> **This uploads your audio.** MAUS runs at the Bavarian Archive for Speech
> Signals in Munich; the recordings and their transcripts leave your machine. It
> is an academic service, not a commercial one, but it is still a third party.
> For the material in this project that was an acceptable trade for a comparison
> baseline. **For recordings your speakers did not consent to share, it is not.**
> Everything else in this repository runs locally.

```bash
curl -X POST -F SIGNAL=@utt.wav -F BPF=@utt.par -F LANGUAGE=deu-DE -F MODUS=align \
  -F OUTFORMAT=TextGrid \
  https://clarin.phonetik.uni-muenchen.de/BASWebServices/services/runMAUS
```

### Reading the comparison

With no hand-corrected reference, "which aligner is right" is not answerable.
What the numbers above report is the **three-way spread** — how far apart the
three systems place the same boundary — plus each system's deviation from the
median of the three. That median leans toward MFA and MAUS, which share an
HMM-GMM lineage, so a wav2vec2-specific improvement is *understated* by it. Treat
these as agreement, not accuracy, until a hand-corrected subset exists.


In [ ]:
# A run-anywhere check that the recommended default is doing what it claims.
_p, _w, _d = al.align(man.iloc[0]["audio_path"], man.iloc[0]["text"], mode="vc")
assert _p[0]["start"] >= 0 and abs(_p[-1]["end"] - _w[-1]["end"]) < 1e-6
assert all(b["start"] >= a["end"] - 1e-6 for a, b in zip(_p, _p[1:])), "phones overlap"
assert len(_w) == len(str(man.iloc[0]["text"]).split()), "word count drifted"
print(f"OK — {len(_p)} phones, {len(_w)} words, {_d:.2f}s, monotonic")
